In [2]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, json, shutil, glob, re
import numpy as np
import pandas as pd

BASE_DIR = Path("/content/drive/MyDrive/EGFR_DESKTOP_AI_APP")
DATA_DIR = BASE_DIR / "data"
MODEL_DIR = BASE_DIR / "models"
EXPORT_DIR = BASE_DIR / "exports"
APP_DIR = BASE_DIR / "desktop_app"

for d in [DATA_DIR, MODEL_DIR, EXPORT_DIR, APP_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Workspace created:")
print(BASE_DIR)

Mounted at /content/drive
Workspace created:
/content/drive/MyDrive/EGFR_DESKTOP_AI_APP


In [3]:
SEARCH_ROOT = Path("/content/drive/MyDrive")

patient_json_files = list(SEARCH_ROOT.rglob("patient_precision_TCGA*.json"))

print("Number of patient precision JSON files found:", len(patient_json_files))

for p in patient_json_files[:20]:
    print(p)

if len(patient_json_files) == 0:
    raise Exception("No patient_precision_TCGA*.json files found. Check your Drive paths.")

Number of patient precision JSON files found: 46
/content/drive/MyDrive/pfe_digital_twin_app/Data/patient_precision /patient_precision_TCGA_67_6217.json
/content/drive/MyDrive/pfe_digital_twin_app/Data/patient_precision /patient_precision_TCGA_91_6847.json
/content/drive/MyDrive/pfe_digital_twin_app/Data/patient_precision /patient_precision_TCGA_44_2661.json
/content/drive/MyDrive/pfe_digital_twin_app/Data/patient_precision /patient_precision_TCGA_05_4402.json
/content/drive/MyDrive/pfe_digital_twin_app/Data/patient_precision /patient_precision_TCGA_64_1681.json
/content/drive/MyDrive/pfe_digital_twin_app/Data/patient_precision /patient_precision_TCGA_69_7765.json
/content/drive/MyDrive/pfe_digital_twin_app/Data/patient_precision /patient_precision_TCGA_50_6595.json
/content/drive/MyDrive/pfe_digital_twin_app/Data/patient_precision /patient_precision_TCGA_49_4494.json
/content/drive/MyDrive/pfe_digital_twin_app/Data/patient_precision /patient_precision_TCGA_38_6178.json
/content/drive/

In [4]:
def safe_get(d, keys, default=None):
    if not isinstance(d, dict):
        return default

    for key in keys:
        if key in d and d[key] not in [None, "", "Unknown", "unknown", "NA", "N/A"]:
            return d[key]

    return default


def normalize_list(x):
    if x is None:
        return []
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        if x.strip() == "":
            return []
        # Split only if it looks like a list
        if "," in x:
            return [i.strip() for i in x.split(",") if i.strip()]
        return [x]
    return [x]


def normalize_number(x, default=np.nan):
    try:
        if x in [None, "", "Unknown", "unknown", "NA", "N/A"]:
            return default
        return float(x)
    except:
        return default


def normalize_stage(stage):
    s = str(stage).upper()

    if "IV" in s or "STAGE IV" in s or "M1" in s:
        return "IV"
    if "III" in s or "STAGE III" in s:
        return "III"
    if "II" in s or "STAGE II" in s:
        return "II"
    if "I" in s or "STAGE I" in s:
        return "I"

    return "Unknown"


def normalize_egfr_class(x):
    s = str(x).lower()

    if "exon 19" in s or "19 deletion" in s or "elrea" in s or "del" in s:
        return "EGFR exon 19 deletion"
    if "l858r" in s:
        return "EGFR L858R"
    if "t790m" in s:
        return "EGFR T790M resistance"
    if "exon 20" in s or "ins" in s:
        return "EGFR exon 20 insertion"
    if "negative" in s or "wild" in s:
        return "EGFR negative"
    if "egfr" in s:
        return "Other EGFR mutation"

    return "Unknown EGFR"


def get_therapy_list(raw):
    therapy_keys = [
        "ranked_therapies",
        "treatments",
        "therapy_ranking",
        "recommended_treatments",
        "ranked_treatments",
        "final_ranked_drugs",
        "drug_ranking"
    ]

    therapies = safe_get(raw, therapy_keys, [])

    if isinstance(therapies, dict):
        therapies = list(therapies.values())

    if not isinstance(therapies, list):
        therapies = []

    clean = []

    for i, t in enumerate(therapies):
        if isinstance(t, dict):
            drug = safe_get(t, ["drug", "name", "treatment", "therapy"], "Unknown")
            eff = normalize_number(safe_get(t, ["eff", "efficiency", "predicted_efficiency", "score"], 0.7), 0.7)
            res = normalize_number(safe_get(t, ["res", "resistance", "resistance_risk"], 0.4), 0.4)

            if eff > 1:
                eff = eff / 100
            if res > 1:
                res = res / 100

            clean.append({
                "rank": int(safe_get(t, ["rank"], i + 1)),
                "drug": drug,
                "efficiency": float(np.clip(eff, 0, 1)),
                "resistance": float(np.clip(res, 0, 1))
            })

        elif isinstance(t, str):
            clean.append({
                "rank": i + 1,
                "drug": t,
                "efficiency": max(0.35, 0.85 - i * 0.05),
                "resistance": min(0.85, 0.25 + i * 0.06)
            })

    return clean


DEFAULT_DRUG_LIBRARY = [
    "Osimertinib",
    "Lazertinib",
    "Amivantamab",
    "Amivantamab + Lazertinib",
    "Osimertinib + Chemotherapy",
    "Afatinib",
    "Dacomitinib",
    "Gefitinib",
    "Erlotinib",
    "Platinum-based Chemotherapy"
]


def default_therapies_for_patient(egfr_class, has_met, has_tp53):
    therapies = []

    for i, drug in enumerate(DEFAULT_DRUG_LIBRARY):
        base_eff = 0.55
        base_res = 0.45

        if egfr_class in ["EGFR exon 19 deletion", "EGFR L858R"]:
            if drug == "Osimertinib":
                base_eff = 0.86
                base_res = 0.34
            elif drug == "Osimertinib + Chemotherapy":
                base_eff = 0.88
                base_res = 0.31
            elif drug == "Amivantamab + Lazertinib":
                base_eff = 0.90 if has_met else 0.82
                base_res = 0.25 if has_met else 0.36
            elif drug in ["Afatinib", "Dacomitinib"]:
                base_eff = 0.74
                base_res = 0.48
            elif drug in ["Gefitinib", "Erlotinib"]:
                base_eff = 0.68
                base_res = 0.55
            elif drug == "Platinum-based Chemotherapy":
                base_eff = 0.61
                base_res = 0.52

        elif egfr_class == "EGFR T790M resistance":
            if drug == "Osimertinib":
                base_eff = 0.82
                base_res = 0.42
            elif drug == "Amivantamab + Lazertinib":
                base_eff = 0.78
                base_res = 0.38
            elif drug == "Platinum-based Chemotherapy":
                base_eff = 0.58
                base_res = 0.57

        elif egfr_class == "EGFR exon 20 insertion":
            if "Amivantamab" in drug:
                base_eff = 0.74
                base_res = 0.44
            elif drug == "Platinum-based Chemotherapy":
                base_eff = 0.60
                base_res = 0.55
            else:
                base_eff = 0.45
                base_res = 0.65

        elif egfr_class == "EGFR negative":
            if drug == "Platinum-based Chemotherapy":
                base_eff = 0.62
                base_res = 0.55
            else:
                base_eff = 0.35
                base_res = 0.70

        if has_tp53:
            base_res += 0.08
            base_eff -= 0.03

        therapies.append({
            "rank": i + 1,
            "drug": drug,
            "efficiency": float(np.clip(base_eff, 0.05, 1.0)),
            "resistance": float(np.clip(base_res, 0.0, 1.0))
        })

    therapies = sorted(therapies, key=lambda x: (-x["efficiency"], x["resistance"]))

    for i, t in enumerate(therapies):
        t["rank"] = i + 1

    return therapies


patient_rows = []
therapy_rows = []
raw_patients = {}

for fp in patient_json_files:
    with open(fp, "r", encoding="utf-8") as f:
        raw = json.load(f)

    pid = safe_get(raw, ["patient_id", "id", "case_id"], fp.stem.replace("patient_precision_", ""))

    age = normalize_number(safe_get(raw, ["age", "Age", "age_at_diagnosis", "Age at diagnosis"], np.nan))
    sex = safe_get(raw, ["sex", "gender", "Sex", "Gender"], "Unknown")
    stage = normalize_stage(safe_get(raw, ["stage", "tumor_stage", "Tumor stage", "ajcc_pathologic_stage", "ajcc_pathologic_t"], "Unknown"))
    survival_status = safe_get(raw, ["survival_status", "vital_status", "status"], "Unknown")

    egfr_raw = safe_get(raw, [
        "primary_egfr", "raw_egfr", "raw_egfr_mutation",
        "egfr_status", "egfr", "EGFR", "EGFR_mutation", "mutation"
    ], "Unknown")

    egfr_class = normalize_egfr_class(egfr_raw)

    secondary = normalize_list(safe_get(raw, [
        "secondary_mutations",
        "secondary_alterations",
        "co_mutations",
        "alterations"
    ], []))

    resistance_mechanisms = normalize_list(safe_get(raw, [
        "resistance_mechanisms",
        "resistance"
    ], []))

    has_tp53 = int(any("TP53" in str(x).upper() for x in secondary))
    has_met = int(any("MET" in str(x).upper() for x in secondary))
    has_kras = int(any("KRAS" in str(x).upper() for x in secondary))
    has_pik3ca = int(any("PIK3CA" in str(x).upper() for x in secondary))
    has_bypass = int(any("bypass" in str(x).lower() for x in resistance_mechanisms))
    has_resistance = int(len(resistance_mechanisms) > 0)

    # Optional scores if already exist in your JSON
    egfr_pathway_activity = normalize_number(safe_get(raw, [
        "egfr_pathway_activity", "EGFR_pathway_activity", "pathway_activity"
    ], np.nan))

    tumor_aggressiveness = normalize_number(safe_get(raw, [
        "tumor_aggressiveness", "aggressiveness_score", "tumor_aggressiveness_score"
    ], np.nan))

    pathology_score = normalize_number(safe_get(raw, [
        "pathology_score", "pathology_risk", "qupath_score", "cellularity_score"
    ], np.nan))

    immune_score = normalize_number(safe_get(raw, [
        "immune_score", "immune_activity", "immune_microenvironment_score"
    ], np.nan))

    therapies = get_therapy_list(raw)

    if len(therapies) == 0:
        therapies = default_therapies_for_patient(egfr_class, has_met, has_tp53)

    best = sorted(therapies, key=lambda x: x["rank"])[0]

    patient_record = {
        "patient_id": pid,
        "age": age,
        "sex": sex,
        "stage": stage,
        "survival_status": survival_status,
        "egfr_raw": egfr_raw,
        "egfr_class": egfr_class,
        "secondary_mutations": secondary,
        "resistance_mechanisms": resistance_mechanisms,
        "has_TP53": has_tp53,
        "has_MET": has_met,
        "has_KRAS": has_kras,
        "has_PIK3CA": has_pik3ca,
        "has_bypass": has_bypass,
        "has_resistance_mechanism": has_resistance,
        "egfr_pathway_activity": egfr_pathway_activity,
        "tumor_aggressiveness": tumor_aggressiveness,
        "pathology_score": pathology_score,
        "immune_score": immune_score,
        "best_known_treatment": best["drug"],
        "best_known_efficiency": best["efficiency"],
        "best_known_resistance": best["resistance"]
    }

    patient_rows.append(patient_record)

    for t in therapies:
        therapy_rows.append({
            **patient_record,
            "drug": t["drug"],
            "rank": t["rank"],
            "target_efficiency": t["efficiency"],
            "target_resistance": t["resistance"],
            "is_best_treatment": int(t["rank"] == 1)
        })

    raw_patients[pid] = raw


patients_df = pd.DataFrame(patient_rows)
therapy_df = pd.DataFrame(therapy_rows)

# Fill numerical missing values
for col in ["age", "egfr_pathway_activity", "tumor_aggressiveness", "pathology_score", "immune_score"]:
    if col in patients_df.columns:
        median = patients_df[col].median() if patients_df[col].notna().any() else 0.5
        patients_df[col] = patients_df[col].fillna(median)
        therapy_df[col] = therapy_df[col].fillna(median)

patients_df.to_csv(DATA_DIR / "patients_model_table.csv", index=False)
therapy_df.to_csv(DATA_DIR / "patient_treatment_training_table.csv", index=False)

print("Patients table:", patients_df.shape)
print("Patient-treatment table:", therapy_df.shape)

display(patients_df.head())
display(therapy_df.head())

Patients table: (46, 22)
Patient-treatment table: (460, 27)


,patient_id,age,sex,stage,survival_status,egfr_raw,egfr_class,secondary_mutations,resistance_mechanisms,has_TP53,...,has_PIK3CA,has_bypass,has_resistance_mechanism,egfr_pathway_activity,tumor_aggressiveness,pathology_score,immune_score,best_known_treatment,best_known_efficiency,best_known_resistance
0,TCGA-67-6217,0.5,Unknown,Unknown,Unknown,Unknown,Unknown EGFR,[],[],0,...,0,0,0,0.5,0.5,0.5,0.5,Osimertinib,0.55,0.45
1,TCGA-91-6847,0.5,Unknown,Unknown,Unknown,Unknown,Unknown EGFR,[],[],0,...,0,0,0,0.5,0.5,0.5,0.5,Osimertinib,0.55,0.45
2,TCGA-44-2661,0.5,Unknown,Unknown,Unknown,Unknown,Unknown EGFR,[],[],0,...,0,0,0,0.5,0.5,0.5,0.5,Osimertinib,0.55,0.45
3,TCGA-05-4402,0.5,Unknown,Unknown,Unknown,Unknown,Unknown EGFR,[],[],0,...,0,0,0,0.5,0.5,0.5,0.5,Osimertinib,0.55,0.45
4,TCGA-64-1681,0.5,Unknown,Unknown,Unknown,Unknown,Unknown EGFR,[],[],0,...,0,0,0,0.5,0.5,0.5,0.5,Osimertinib,0.55,0.45


,patient_id,age,sex,stage,survival_status,egfr_raw,egfr_class,secondary_mutations,resistance_mechanisms,has_TP53,...,pathology_score,immune_score,best_known_treatment,best_known_efficiency,best_known_resistance,drug,rank,target_efficiency,target_resistance,is_best_treatment
0,TCGA-67-6217,0.5,Unknown,Unknown,Unknown,Unknown,Unknown EGFR,[],[],0,...,0.5,0.5,Osimertinib,0.55,0.45,Osimertinib,1,0.55,0.45,1
1,TCGA-67-6217,0.5,Unknown,Unknown,Unknown,Unknown,Unknown EGFR,[],[],0,...,0.5,0.5,Osimertinib,0.55,0.45,Lazertinib,2,0.55,0.45,0
2,TCGA-67-6217,0.5,Unknown,Unknown,Unknown,Unknown,Unknown EGFR,[],[],0,...,0.5,0.5,Osimertinib,0.55,0.45,Amivantamab,3,0.55,0.45,0
3,TCGA-67-6217,0.5,Unknown,Unknown,Unknown,Unknown,Unknown EGFR,[],[],0,...,0.5,0.5,Osimertinib,0.55,0.45,Amivantamab + Lazertinib,4,0.55,0.45,0
4,TCGA-67-6217,0.5,Unknown,Unknown,Unknown,Unknown,Unknown EGFR,[],[],0,...,0.5,0.5,Osimertinib,0.55,0.45,Osimertinib + Chemotherapy,5,0.55,0.45,0


In [6]:
# Cellule vidée pour corriger l'erreur de syntaxe.